# How to Win Jeopardy

## Introduction
Jeopardy is a popular TV show in the US where participants answer questions to win money. Imagine that you want to compete on Jeopardy, and you're looking for any way to win. In this project, we'll work with a dataset of Jeopardy questions to figure out some patterns in the questions that could help you win.

The dataset contains 20,000 rows of Jeopardy questions dated back to 12/31/2004. The full dataset(JSON or CSV) can be found at [this link](https://www.reddit.com/r/datasets/comments/1uyd0t/200000_jeopardy_questions_in_a_json_file/)

In [62]:
import pandas as pd
from random import choice
from scipy.stats import chisquare
import numpy as np

In [3]:
df=pd.read_csv("C:/Users/Public/Documents/yang/Data Science Project/data/JEOPARDY_CSV.csv")

In [4]:
df.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216930 entries, 0 to 216929
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Show Number  216930 non-null  int64 
 1    Air Date    216930 non-null  object
 2    Round       216930 non-null  object
 3    Category    216930 non-null  object
 4    Value       216930 non-null  object
 5    Question    216930 non-null  object
 6    Answer      216928 non-null  object
dtypes: int64(1), object(6)
memory usage: 11.6+ MB


After reviewing the dataset briefly:
- we found out that some columns have spaces in front. For the better data processing, let's remove spaces from columns's names
- There are two null values in 'Answer' column. We will remove these two rows.

In [6]:
df.columns

Index(['Show Number', ' Air Date', ' Round', ' Category', ' Value',
       ' Question', ' Answer'],
      dtype='object')

In [7]:
#removing space from columns' name
columns=[col.strip() for col in df.columns]

In [8]:
df.columns=columns

In [9]:
df.columns

Index(['Show Number', 'Air Date', 'Round', 'Category', 'Value', 'Question',
       'Answer'],
      dtype='object')

In [10]:
#checking the null values
df[df['Answer'].isnull()]

,Show Number,Air Date,Round,Category,Value,Question,Answer
94817,4346,2003-06-23,Jeopardy!,"GOING ""N""SANE",$200,"It often precedes ""and void""",NaN
143297,6177,2011-06-21,Double Jeopardy!,NOTHING,$400,"This word for ""nothing"" precedes ""and void"" to...",NaN


In [11]:
#Drop two rows with null values
df.dropna(inplace=True)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 216928 entries, 0 to 216929
Data columns (total 7 columns):
 #   Column       Non-Null Count   Dtype 
---  ------       --------------   ----- 
 0   Show Number  216928 non-null  int64 
 1   Air Date     216928 non-null  object
 2   Round        216928 non-null  object
 3   Category     216928 non-null  object
 4   Value        216928 non-null  object
 5   Question     216928 non-null  object
 6   Answer       216928 non-null  object
dtypes: int64(1), object(6)
memory usage: 13.2+ MB


## Normalize Texts and Values

The best way to analyze the questions and answers is by tokens or words. Therefore, we need to convert the sentences into individual words. We will remove the punctuations from the texts, put them in lowercase, and split them into words.We can call this process as text normalization.

We will write a function to normalize the questions and answers

In [13]:
df_clean=df.copy().reset_index(drop=True)

In [14]:
df_clean.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams


In [15]:
#Normalize the text
import re

def text_norm(string):#take a string input
    txt=re.sub('\W',' ',string) #remove all non-word characters and replace with a white space
    txt=txt.lower()

    
    return txt

In [16]:
#apply the function to 'Question' column and assign the result to 'Clean_Question'

df_clean['Clean_Question']=df_clean['Question'].apply(text_norm)

In [17]:
#apply the function to 'Answer' column and assign the result to 'Clean_Answer'
df_clean['Clean_Answer']=df_clean['Answer'].apply(text_norm)

In [18]:
df_clean.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean_Question,Clean_Answer
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus,for the last 8 years of his life galileo was ...,copernicus
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe,no 2 1912 olympian football star at carlisl...,jim thorpe
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona,the city of yuma in this state has a record av...,arizona
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's,in 1963 live on the art linkletter show th...,mcdonald s
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams,signer of the dec of indep framer of the co...,john adams


In [19]:
df_clean.shape

(216928, 9)

We also notice some issues with 'Air Date' and 'Value' columns. In order to work dataset easier:
- We will remove "$" from the begining of each value and convert the column to numeric in 'Value' column
- We will convert 'Air Date' to datetime object

In [20]:
df_clean['Value'].unique()

array(['$200', '$400', '$600', '$800', '$2,000', '$1000', '$1200',
       '$1600', '$2000', '$3,200', 'None', '$5,000', '$100', '$300',
       '$500', '$1,000', '$1,500', '$1,200', '$4,800', '$1,800', '$1,100',
       '$2,200', '$3,400', '$3,000', '$4,000', '$1,600', '$6,800',
       '$1,900', '$3,100', '$700', '$1,400', '$2,800', '$8,000', '$6,000',
       '$2,400', '$12,000', '$3,800', '$2,500', '$6,200', '$10,000',
       '$7,000', '$1,492', '$7,400', '$1,300', '$7,200', '$2,600',
       '$3,300', '$5,400', '$4,500', '$2,100', '$900', '$3,600', '$2,127',
       '$367', '$4,400', '$3,500', '$2,900', '$3,900', '$4,100', '$4,600',
       '$10,800', '$2,300', '$5,600', '$1,111', '$8,200', '$5,800',
       '$750', '$7,500', '$1,700', '$9,000', '$6,100', '$1,020', '$4,700',
       '$2,021', '$5,200', '$3,389', '$4,200', '$5', '$2,001', '$1,263',
       '$4,637', '$3,201', '$6,600', '$3,700', '$2,990', '$5,500',
       '$14,000', '$2,700', '$6,400', '$350', '$8,600', '$6,300', '$250',
    

After checking the unique values in 'Value' column, we find out there are 'None' values. When we treat the column, we will convert 'None' to 0.Otherwise it will raise Exception Errors when converting the datatype to integer.

In [21]:
# remove all signs and convert 'Value' column to numeric
def value_norm(text):
    text=re.sub("\W","",text)# replace '$'and ',' with ''
    try:
        text=int(text)
    except Exception:  #Exception for 'None' values
        text=0
    
    return text
        
    

In [22]:
df_clean['Clean_value']=df_clean['Value'].apply(value_norm)

In [23]:
df_clean.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean_Question,Clean_Answer,Clean_value
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus,for the last 8 years of his life galileo was ...,copernicus,200
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe,no 2 1912 olympian football star at carlisl...,jim thorpe,200
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona,the city of yuma in this state has a record av...,arizona,200
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's,in 1963 live on the art linkletter show th...,mcdonald s,200
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams,signer of the dec of indep framer of the co...,john adams,200


In [24]:
#convert 'Air Date' to datetime object
df_clean['Date']=pd.to_datetime(df_clean['Air Date'])

In [25]:
df_clean.head()

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean_Question,Clean_Answer,Clean_value,Date
0,4680,2004-12-31,Jeopardy!,HISTORY,$200,"For the last 8 years of his life, Galileo was ...",Copernicus,for the last 8 years of his life galileo was ...,copernicus,200,2004-12-31
1,4680,2004-12-31,Jeopardy!,ESPN's TOP 10 ALL-TIME ATHLETES,$200,No. 2: 1912 Olympian; football star at Carlisl...,Jim Thorpe,no 2 1912 olympian football star at carlisl...,jim thorpe,200,2004-12-31
2,4680,2004-12-31,Jeopardy!,EVERYBODY TALKS ABOUT IT...,$200,The city of Yuma in this state has a record av...,Arizona,the city of yuma in this state has a record av...,arizona,200,2004-12-31
3,4680,2004-12-31,Jeopardy!,THE COMPANY LINE,$200,"In 1963, live on ""The Art Linkletter Show"", th...",McDonald's,in 1963 live on the art linkletter show th...,mcdonald s,200,2004-12-31
4,4680,2004-12-31,Jeopardy!,EPITAPHS & TRIBUTES,$200,"Signer of the Dec. of Indep., framer of the Co...",John Adams,signer of the dec of indep framer of the co...,john adams,200,2004-12-31


In [26]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 216928 entries, 0 to 216927
Data columns (total 11 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   Show Number     216928 non-null  int64         
 1   Air Date        216928 non-null  object        
 2   Round           216928 non-null  object        
 3   Category        216928 non-null  object        
 4   Value           216928 non-null  object        
 5   Question        216928 non-null  object        
 6   Answer          216928 non-null  object        
 7   Clean_Question  216928 non-null  object        
 8   Clean_Answer    216928 non-null  object        
 9   Clean_value     216928 non-null  int64         
 10  Date            216928 non-null  datetime64[ns]
dtypes: datetime64[ns](1), int64(2), object(8)
memory usage: 18.2+ MB


## Answers in Questions

In order to figure out whether to study past questions, study general knowledge, or not study it all, it would be helpful to figure out two things:
- How often the answer can be used for a question
- How often questions are repeated(How often complex words(>6 characters) reoccur.)

First, let's examine how often the answer was used for a question. We will write a function to count how many times answers occured in questions.

In [27]:
def count_answer(row):
    split_answer=row['Clean_Answer'].split()
    split_question=row['Clean_Question'].split()
    match_count=0
    if 'the' in split_answer:          #'the' is one of the most common words that doesn't have actual meaning.Therefore, remove 'the' in the answers
        split_answer.remove('the')
    if len(split_answer)==0:
        return 0
    for item in split_answer:
        if item in split_question:
            match_count +=1
    return match_count/len(split_answer)

In [28]:
df_clean['answer_in_question']=df_clean.apply(count_answer,axis=1)


In [29]:
df_clean['answer_in_question'].mean()

0.06141517294157886

There are only 6% answers appearing in the questions. We can't rely on hearing questions to determine the answer. We need to study more.

## Repeated Questions
Let's investigate how often new questions are repeats of older ones.
We will use the following algorithm to fullfill the objective:
- Sort jeopardy in order of ascending air date.
- Maintain a set called terms_used that will be empty initially.
- Iterate through each row of jeopardy.
- Split clean_question into words, remove any word shorter than 6 characters, and check if each word occurs in terms_used.
   - If it does, increment a counter.
   - Add each word to terms_used.

In [30]:
question_repeated=[] # a list to store the repeated percentage of terms used in previous questions
terms_used=set() # initiate set() to make the used word unique

df_clean=df_clean.sort_values('Date') # sort the questions by Date so we can compare the new ones against old ones.

for i, row in df_clean.iterrows():
    split_question=row['Clean_Question'].split()
    split_question=[q for q in split_question if len(q) > 5]
    match_count=0
    for word in split_question: #checking if terms in newer questions were used in older questions
        if word in terms_used:
            match_count +=1
    for word in split_question: #store the used terms in the set()
        terms_used.add(word)
    if len(split_question)>0:
        match_count /= len(split_question)# calculating the percentage of repetitions from previous questions in the current questions
    question_repeated.append(match_count)

df_clean['pct_repeated_Q']=question_repeated

In [31]:
df_clean['pct_repeated_Q'].mean()

0.8987545560029684

On average, 89.87% of words were repetitions from previous questions. It appears that we can study the previous questions. However, this number only indicates the single word is repeated from all words used in the previous questions.It doesn't necessarily mean the current question is similar to the previous questions.

Let's study another patterns : the values of questions.

## Whether the used words are associated with the value of questions
The questions with high value will definitely help us win more money. If certain used words appeared more in the high valued questions, we could study more on those words for higher rewards.

First, let's investigate whether the used words are associated with the type of questions in terms of prize value using a chi-squared test. If there is an association, we can explore what words occured more in the high valued question.

In [51]:
#define high-valued questions
df_clean['Clean_value'].median()

600.0

In [33]:
df_clean['Clean_value'].mean()

739.9925320843782

Since the average value of questions is 740, let's say the questions over $750 are high-valued.

In [34]:
df_clean['high_value']=df_clean['Clean_value'].apply(lambda x: 1 if x > 750 else 0)

In [35]:
df_clean.head(10)

,Show Number,Air Date,Round,Category,Value,Question,Answer,Clean_Question,Clean_Answer,Clean_value,Date,answer_in_question,pct_repeated_Q,high_value
84523,1,1984-09-10,Jeopardy!,LAKES & RIVERS,$100,River mentioned most often in the Bible,the Jordan,river mentioned most often in the bible,the jordan,100,1984-09-10,0.000000,0.0,0
84565,1,1984-09-10,Double Jeopardy!,THE BIBLE,$1000,"According to 1st Timothy, it is the ""root of a...",the love of money,according to 1st timothy it is the root of a...,the love of money,1000,1984-09-10,0.333333,0.0,1
84566,1,1984-09-10,Double Jeopardy!,'50'S TV,$1000,Name under which experimenter Don Herbert taug...,Mr. Wizard,name under which experimenter don herbert taug...,mr wizard,1000,1984-09-10,0.000000,0.0,1
84567,1,1984-09-10,Double Jeopardy!,NATIONAL LANDMARKS,$1000,D.C. building shaken by November '83 bomb blast,the Capitol,d c building shaken by november 83 bomb blast,the capitol,1000,1984-09-10,0.000000,0.0,1
84568,1,1984-09-10,Double Jeopardy!,NOTORIOUS,$1000,"After the deed, he leaped to the stage shoutin...",John Wilkes Booth,after the deed he leaped to the stage shoutin...,john wilkes booth,1000,1984-09-10,0.000000,0.0,1
84569,1,1984-09-10,Double Jeopardy!,4-LETTER WORDS,$1000,The president takes one before stepping into o...,oath,the president takes one before stepping into o...,oath,1000,1984-09-10,0.000000,0.0,1
84570,1,1984-09-10,Final Jeopardy!,HOLIDAYS,None,The third Monday of January starting in 1986,Martin Luther King Day,the third monday of january starting in 1986,martin luther king day,0,1984-09-10,0.000000,0.0,0
84538,1,1984-09-10,Jeopardy!,LAKES & RIVERS,$400,American river only 33 miles shorter than the ...,the Missouri,american river only 33 miles shorter than the ...,the missouri,400,1984-09-10,0.000000,0.0,0
84537,1,1984-09-10,Jeopardy!,ACTORS & ROLES,$300,"He may ""Never Say Never Again"" when asked to b...",Sean Connery,he may never say never again when asked to b...,sean connery,300,1984-09-10,0.000000,0.0,0
84536,1,1984-09-10,Jeopardy!,FOREIGN CUISINE,$300,Jewish crepe filled with cheese,a blintz,jewish crepe filled with cheese,a blintz,300,1984-09-10,0.000000,0.0,0


We will write a function to count the frequencies of given words occuring in the high and low valued questions

In [52]:
def count_values(word):
    low_count=0
    high_count=0
    for i, row in df_clean.iterrows():
        if word in row['Clean_Question'].split():
            if row['high_value']==1:
                high_count +=1
            else:
                low_count +=1
                
                
    return high_count,low_count

In [56]:
# take 10 random sample from the words_used_List to investigate the chi-squared test

terms_used_list=list(terms_used)
comparison_sample=[choice(terms_used_list) for i in range(10)]

observed_count=[]

for word in comparison_sample:
    observed_count.append(count_values(word))
    
    

In [57]:
observed_count

[(1, 2),
 (0, 4),
 (0, 1),
 (0, 1),
 (1, 0),
 (1, 0),
 (0, 2),
 (0, 1),
 (0, 1),
 (2, 7)]

In [60]:

high_value_count=df_clean[df_clean['high_value']==1].shape[0]
low_value_count=df_clean[df_clean['high_value']==0].shape[0]

chi_squared=[]
for obs in observed_count:
    total=sum(obs)
    total_prop=total/len(df_clean)
    high_exp=total_prop * high_value_count
    low_exp=total_prop * low_value_count
    
    observed=np.array([obs[0],obs[1]])
    expected=np.array([high_exp,low_exp])
    chi_squared.append(chisquare(observed,expected))

In [61]:
chi_squared

[Power_divergenceResult(statistic=0.1144170942544174, pvalue=0.7351703087318446),
 Power_divergenceResult(statistic=3.0177686117513844, pvalue=0.0823567112987689),
 Power_divergenceResult(statistic=0.7544421529378461, pvalue=0.3850734529454555),
 Power_divergenceResult(statistic=0.7544421529378461, pvalue=0.3850734529454555),
 Power_divergenceResult(statistic=1.3254826710118672, pvalue=0.2496104480108039),
 Power_divergenceResult(statistic=1.3254826710118672, pvalue=0.2496104480108039),
 Power_divergenceResult(statistic=1.5088843058756922, pvalue=0.21930941035235324),
 Power_divergenceResult(statistic=0.7544421529378461, pvalue=0.3850734529454555),
 Power_divergenceResult(statistic=0.7544421529378461, pvalue=0.3850734529454555),
 Power_divergenceResult(statistic=1.5855106864446595, pvalue=0.2079687058772109)]

The p-values indicate that there are not significant relationship between words used(from the random sample) and values of questions. 

## Category of Questions vs. Values

In [48]:
df_clean.loc[df_clean['high_value']==1,'Category'].value_counts(normalize=True)

BEFORE & AFTER                0.003259
LITERATURE                    0.002155
SCIENCE                       0.002123
OPERA                         0.001608
WORD ORIGINS                  0.001512
                                ...   
BROAD WEIGH                   0.000011
THE TOP TEN OF EVERYTHING     0.000011
MAD ABOUT MADAGASCAR          0.000011
FARAWAY PLACES                0.000011
NO LONGER AN OLYMPIC SPORT    0.000011
Name: Category, Length: 23071, dtype: float64

In [50]:
df_clean['Category'].value_counts(normalize=True)

BEFORE & AFTER                0.002522
SCIENCE                       0.002392
LITERATURE                    0.002286
AMERICAN HISTORY              0.001927
POTPOURRI                     0.001849
                                ...   
MEDIEVAL FIRSTS               0.000005
SCIENTIFIC THEORIES           0.000005
ENGLISH WRITERS               0.000005
NO. 1 HITS OF THE 1970s       0.000005
2000 PRESIDENTIAL HOPEFULS    0.000005
Name: Category, Length: 27995, dtype: float64

It appears that the topics of 'BEFORE & AFTER','LITERATURE', and 'Science' occured more often than other topics, especially 'BEFORE & AFTER' in high-valued questions.

## Summary
After our analysis, we can conclude:
- There are very few answers appearing in the corresponding questions
- There are about 89.78% of individual words appearing in the previous questions. Therefore, studying the previous questions might improve your chance to win
- It seems there is no significantly relationship between words used and values of questions
- Studying more questions in the topics of 'BEFORE & AFTER','LITERATURE', and 'Science' might help win high-valued questions